**Zwischenschritt**
In diesem Schritt will ich nochmal schauen, wie sehr denn die Feinstaubwerte der Sensoren miteinander korrelieren. Es ist nicht von einer perfekten Korrelation auszugehene, da die intraurbanen Unterschiede vorhanden sind und es ja auch das Ziel dieser Analyse ist, die verschiedenen Werte herauszufinden, aber es eine Großwetterlage sollte trotzdem die Sensoren in eine ähnliche Richtung beeinflussen.

In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# =========================================================
# 1. Pfade definieren
# =========================================================
file_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"
output_dir = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Paper\IEEE-conference-template-062824\figures"

os.makedirs(output_dir, exist_ok=True)
save_path_pdf = os.path.join(output_dir, "sensor_correlation_matrix.pdf")
save_path_png = os.path.join(output_dir, "sensor_correlation_matrix.png")

# =========================================================
# 2. Daten laden und vorbereiten
# =========================================================
print("Lade Masterdatensatz für die Qualitätsprüfung...")
df = pd.read_csv(file_path, sep=";", decimal=",")

df["station_id"] = df["station_id"].astype(str).str.strip()

# =========================================================
# 3. Datensatz umstrukturieren (Pivot)
# =========================================================
df_pivot = df.pivot_table(
    index="date", columns="station_id", values="PM25_Target"
)
print(
    f"Datenstruktur erstellt. Anzahl gemeinsamer Messtage im Zeitraum: {len(df_pivot)}"
)

# =========================================================
# 4. Korrelationsmatrix berechnen und umsortieren
# =========================================================
corr_matrix = df_pivot.corr()

# Gewünschte Sortierung definieren
start_sensor = ['1']
end_sensors = ['9094', '46505', '15449', '5456']
middle_sensors = [col for col in corr_matrix.columns if col not in start_sensor + end_sensors]

desired_order = start_sensor + middle_sensors + end_sensors

# Matrix auf die neue Reihenfolge anpassen (ignoriert Sensoren, falls diese im Datensatz fehlen)
valid_order = [col for col in desired_order if col in corr_matrix.columns]
corr_matrix = corr_matrix.reindex(index=valid_order, columns=valid_order)

# =========================================================
# 5. Spalten und Indizes umbenennen
# =========================================================
rename_mapping = {
    '1': 'AS',
    '9094': 'Group 1',
    '46505': 'Group 1',
    '15449': 'Group 2',
    '5456': 'Group 2'
}
corr_matrix = corr_matrix.rename(index=rename_mapping, columns=rename_mapping)

# =========================================================
# 6. Visuelle Darstellung (Heatmap)
# =========================================================
fig, ax = plt.subplots(figsize=(10, 8))

fig.patch.set_alpha(0.0)
ax.patch.set_alpha(0.0)

heatmap = sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=0.3,
    vmax=1.0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Pearson Correlation Coefficient"},
)

ax.set_title(
    r"Correlation matrix $\mathrm{PM}_{2.5}$-time series",
    fontsize=16,
    fontweight="bold",
    pad=15,
    color="black",
)
ax.set_xlabel("Sensor ID", fontsize=12, color="black")
ax.set_ylabel("Sensor ID", fontsize=12, color="black")

ax.tick_params(colors="black", which="both")
plt.xticks(rotation=45, color="black")
plt.yticks(rotation=0, color="black")

cbar = heatmap.collections[0].colorbar
cbar.ax.yaxis.label.set_color("black")
cbar.ax.tick_params(colors="black")

plt.tight_layout()

# =========================================================
# Einzelne Sensor-IDs farblich hervorheben
# =========================================================
sensor_colors = {
    "Group 1": "red",
    "Group 2": "red"
}

# X-Achse einfärben
for label in ax.get_xticklabels():
    sensor_name = label.get_text()
    label.set_color(sensor_colors.get(sensor_name, "black"))

# Y-Achse einfärben
for label in ax.get_yticklabels():
    sensor_name = label.get_text()
    label.set_color(sensor_colors.get(sensor_name, "black"))

# =========================================================
# 7. Speichern und Anzeigen
# =========================================================
plt.savefig(save_path_pdf, format="pdf", transparent=True, bbox_inches="tight")
plt.savefig(save_path_png, format="png", transparent=True, dpi=300, bbox_inches="tight")

print(f"Erfolgreich gespeichert unter:\n- {save_path_pdf}\n- {save_path_png}")

plt.show()

**Ohne 5456 und 15449**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# 1. Pfad zur Masterdatei
# =========================================================
file_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"

# =========================================================
# 2. Daten laden und vorbereiten
# =========================================================
print("Lade Masterdatensatz für die Qualitätsprüfung...")
df = pd.read_csv(file_path, sep=";", decimal=",")

# Sicherstellen, dass die station_id ein sauberer Text ist
df['station_id'] = df['station_id'].astype(str).str.strip()

# --- NEU: Bestimmte Sensoren (Ausreißer) ausschließen ---
sensoren_zu_entfernen = ['5456', '15449', '9094']
df = df[~df['station_id'].isin(sensoren_zu_entfernen)]
print(f"Sensoren {sensoren_zu_entfernen} wurden für diese Analyse erfolgreich entfernt.")

# =========================================================
# 3. Datensatz umstrukturieren (Pivot)
# =========================================================
# Um eine Korrelation zu berechnen, brauchen wir eine Tabelle: Zeilen=Tage, Spalten=Sensoren
df_pivot = df.pivot_table(index='date', columns='station_id', values='PM25_Target')

print(f"Datenstruktur erstellt. Anzahl gemeinsamer Messtage im Zeitraum: {len(df_pivot)}")

# =========================================================
# 4. Korrelationsmatrix berechnen (Pearson-Korrelation)
# =========================================================
corr_matrix = df_pivot.corr()

# =========================================================
# 5. Visuelle Darstellung (Heatmap)
# =========================================================
plt.figure(figsize=(10, 8))

# Seaborn Heatmap plotten
sns.heatmap(corr_matrix,
            annot=True,  # Schreibt die genauen Korrelationswerte in die Boxen
            cmap='coolwarm',  # Farbschema (Kalt zu Warm)
            fmt=".2f",  # Zwei Nachkommastellen (z.B. 0.85)
            vmin=0.3, vmax=1.0,  # Setzt die Farbskala
            linewidths=0.5,
            cbar_kws={'label': 'Pearson Korrelationskoeffizient'})

# Plot-Styling
plt.title("Korrelationsmatrix der PM2.5-Zeitreihen\n(Ohne Sensoren 5456 & 15449)", fontsize=16, fontweight='bold',
          pad=15)
plt.xlabel("Sensor ID", fontsize=12)
plt.ylabel("Sensor ID", fontsize=12)

# Beschriftungen rotieren, damit sie gut lesbar sind
plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

Hier unten mit dem Offset von 4,88. Macht aber für die Pearson-Korrelation keinen Unterschied.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# 1. Pfad zur Masterdatei
# =========================================================
file_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"

# =========================================================
# 2. Daten laden und vorbereiten
# =========================================================
print("Lade Masterdatensatz für die Qualitätsprüfung...")
df = pd.read_csv(file_path, sep=";", decimal=",")

# Sicherstellen, dass die station_id ein sauberer Text ist
df['station_id'] = df['station_id'].astype(str).str.strip()

# --- Bestimmte Sensoren (Ausreißer) ausschließen ---
sensoren_zu_entfernen = ['5456', '15449', '9094']
df = df[~df['station_id'].isin(sensoren_zu_entfernen)]
print(f"Sensoren {sensoren_zu_entfernen} wurden für diese Analyse erfolgreich entfernt.")

# =========================================================
# 3. Hardware-Kalibrierung (Offset anwenden)
# =========================================================
offset = 4.88
print(f"Führe Baseline-Kalibrierung durch: +{offset} µg/m³ für alle Low-Cost-Sensoren...")

# Erhöhe alle Werte, bei denen die ID NICHT "1" (Kopfklinik) ist
df.loc[df['station_id'] != "1", 'PM25_Target'] += offset

# =========================================================
# 4. Datensatz umstrukturieren (Pivot)
# =========================================================
# Um eine Korrelation zu berechnen, brauchen wir eine Tabelle: Zeilen=Tage, Spalten=Sensoren
df_pivot = df.pivot_table(index='date', columns='station_id', values='PM25_Target')

print(f"Datenstruktur erstellt. Anzahl gemeinsamer Messtage im Zeitraum: {len(df_pivot)}")

# =========================================================
# 5. Korrelationsmatrix berechnen (Pearson-Korrelation)
# =========================================================
corr_matrix = df_pivot.corr()

# =========================================================
# 6. Visuelle Darstellung (Heatmap)
# =========================================================
plt.figure(figsize=(10, 8))

# Seaborn Heatmap plotten
sns.heatmap(corr_matrix,
            annot=True,  # Schreibt die genauen Korrelationswerte in die Boxen
            cmap='coolwarm',  # Farbschema (Kalt zu Warm)
            fmt=".2f",  # Zwei Nachkommastellen (z.B. 0.85)
            vmin=0.3, vmax=1.0,  # Setzt die Farbskala
            linewidths=0.5,
            cbar_kws={'label': 'Pearson Korrelationskoeffizient'})

# Plot-Styling
plt.title("Korrelationsmatrix der PM2.5-Zeitreihen\n(Kalibriert | Ohne Sensoren 5456 & 15449)", fontsize=16,
          fontweight='bold', pad=15)
plt.xlabel("Sensor ID", fontsize=12)
plt.ylabel("Sensor ID", fontsize=12)

# Beschriftungen rotieren, damit sie gut lesbar sind
plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import glob
import os
import numpy as np
import rasterio

folder_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Maps\Einzeltage"
tif_files = glob.glob(os.path.join(folder_path, "*.tif"))

global_min = np.inf
global_max = -np.inf

for tif in tif_files:
    with rasterio.open(tif) as src:
        data = src.read(1)

        # Ungültige Pixel wie NaNs und Nullwerte herausfiltern, damit Ränder die Statistik nicht verfälschen[cite: 1]
        valid_pixels = data[(~np.isnan(data)) & (data > 0)]

        if valid_pixels.size > 0:
            global_min = min(global_min, np.min(valid_pixels))
            global_max = max(global_max, np.max(valid_pixels))

print(f"Absolutes Minimum über alle {len(tif_files)} Dateien: {global_min:.2f}")
print(f"Absolutes Maximum über alle {len(tif_files)} Dateien: {global_max:.2f}")